In [1]:
import pandas as pd
import numpy as np
import zipfile
import matplotlib.pyplot as plt
import gc
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import *
from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings('ignore')

In [2]:
all_data_df = pd.read_csv("Data.csv")
ss = pd.read_csv("SampleSubmission.csv")

### Identify the Phase type patterns
Zindi competition descriptions:

Each consumer device represents a power pole, and the data user associated with the device is identified in the ID column. Some data users are connected to three-phase systems, while others have single-phase connections. This distinction is reflected in the dataset:

*    Single-phase houses have only `v_red`.
*    Three-phase houses include `v_red`, `v_blue`, and `v_yellow`

The dataset includes measured values for voltage, current, power factor, and energy consumption (kWh). These columns are redundant  voltage (V), current(A), power factor(PF) because the are used in the calculation for energy consumption (kWh).

*    Single-Phase: kWh = (V * A * PF) / 1000 * time
*    Three-phase:  kWh = (√3 * V * A * PF) / 1000 * time
    *    V is the difference voltage between the lines
    
The different voltage lines will signify the house type for example if it a residential or commercial building with different power requirements.
    
Only significant data would be the `Source`,`date_time`, `kwh`, `Phase_Type`  

The dataset differs from the described 3 phase and single phase houses having only one voltage input (`v_red`, `v_blue`, or `v_yellow`) per source. Switching from phase type to examining the voltage color used to see if a pattern with these catagorical variables.

In [3]:
def label_colors(df):
    # List of color columns
    color_columns = ['v_red', 'v_blue', 'v_yellow']
    
    # Function to get colors for a row
    def get_colors(row):
        return ', '.join([col.split('_')[1] for col in color_columns if (row[col] != 0 and pd.notna(row[col]))])
    
    # Apply the function to create a new 'v_used' column
    df['v_used'] = df.apply(get_colors, axis=1)
    
    # Function to assign numeric labels
    def assign_label(v_used):
        if v_used == 'red':
            return 0
        elif v_used == 'blue':
            return 1
        elif v_used == 'yellow':
            return 2
        else:
            return -1  # for any other combination or empty string
    
    # Apply the function to create a new 'label' column
    df['v_label'] = df['v_used'].apply(assign_label) # create catagorical integer label
    
    return df

voltage_type = all_data_df.groupby('Source').agg({
    'v_red': 'mean',
    'v_blue': 'mean',
    'v_yellow': 'mean'
}).reset_index()


source_volt = label_colors(voltage_type)
source_volt.head()


,Source,v_red,v_blue,v_yellow,v_used,v_label
0,consumer_device_10_data_user_1,76.443404,NaN,NaN,red,0
1,consumer_device_10_data_user_10,76.443404,NaN,NaN,red,0
2,consumer_device_10_data_user_11,NaN,70.993019,NaN,blue,1
3,consumer_device_10_data_user_12,NaN,NaN,70.8646,yellow,2
4,consumer_device_10_data_user_13,76.443404,NaN,NaN,red,0


In [4]:
source_volt['v_used'].value_counts()

v_used
red       206
yellow    196
blue      183
Name: count, dtype: int64

In [5]:

# Split 'Source' into 'consumer_device_X' and 'data_user_Y'
# increase the catagorical variables for different power use can infer the power generation coming from the consumer_device by looking at all the users who also use that device
all_data_df["consumer_device"] = all_data_df["Source"].str.extract(r'consumer_device_(\d+)_data_user_\d+').astype(int)
all_data_df["data_user"] = all_data_df["Source"].str.extract(r'consumer_device_\d+_data_user_(\d+)').astype(int)

In [6]:
# Assign the voltage type to the corresponding source
source_voltage_dict = source_volt.set_index('Source')['v_label'].to_dict()
all_data_df['v_label'] = all_data_df['Source'].map(source_voltage_dict)

In [7]:
# Display the updated DataFrame (optional)
all_data_df.head()

,date_time,v_red,current,power_factor,kwh,Source,v_blue,v_yellow,consumer_device_9,consumer_device_x,consumer_device,data_user,v_label
0,2024-07-22 18:20:00,137.65,0.08,0.72,0.000661,consumer_device_10_data_user_1,NaN,NaN,0,10,10,1,0
1,2024-07-22 18:25:00,122.82,0.08,0.73,0.000598,consumer_device_10_data_user_1,NaN,NaN,0,10,10,1,0
2,2024-07-22 18:30:00,119.70,0.08,0.74,0.000591,consumer_device_10_data_user_1,NaN,NaN,0,10,10,1,0
3,2024-07-22 18:35:00,124.53,0.08,0.75,0.000623,consumer_device_10_data_user_1,NaN,NaN,0,10,10,1,0
4,2024-07-22 18:40:00,134.84,0.08,0.74,0.000665,consumer_device_10_data_user_1,NaN,NaN,0,10,10,1,0


In [8]:
# Convert 'Datetime' column to datetime objects if it's not already
all_data_df['date_time'] = pd.to_datetime(all_data_df['date_time'])

# Extract the date part For daily aggregation later
all_data_df['Date'] = all_data_df['date_time'].dt.date

## Aggregate to Power data

Aggregate data to hourly to match the resolution of the hourly climate data.

In [9]:
# Find the minimum and maximum date_time values
min_date = all_data_df['Date'].min()
max_date = all_data_df['Date'].max()

print(f"Minimum date_time: {min_date}")
print(f"Maximum date_time: {max_date}")

Minimum date_time: 2023-06-03
Maximum date_time: 2024-09-23


In [10]:
agg_daily = all_data_df.groupby(
    ['Source', 'Date'] # pd.Grouper(key='date_time', freq='D')
).agg({
#     'v_red': 'mean', 
#     'v_blue': 'mean',
#     'v_yellow': 'mean',  # v_label tells if v_red, v_blue, or v_yellow is present
#     'current': 'mean',   # part of kwh equation
#     'power_factor': 'mean', # part of kwh equation
    'kwh': 'sum',
    'v_label': 'first',  # Use first since voltage used are source-consistent
    'consumer_device': 'first',
    'data_user': 'first'
    
}).reset_index()
agg_daily["Date"] = pd.to_datetime(agg_daily["Date"])  # Ensure datetime64[ns]


#### Add Kalam regional Climate data

In [11]:
# Load climate data with datetime parsing
climate_df = pd.read_excel(
    'Climate Data/Kalam Climate Data.xlsx',
    engine='openpyxl',
    parse_dates=['Date Time']
)

In [12]:
# Find the minimum and maximum date_time values
min_date = climate_df['Date Time'].min()
max_date = climate_df['Date Time'].max()

print(f"Minimum date_time: {min_date}")
print(f"Maximum date_time: {max_date}")

Minimum date_time: 2023-06-03 13:00:00
Maximum date_time: 2024-10-25 00:00:00


In [13]:
climate_df['Date'] = climate_df['Date Time'].dt.date

# Aggregate climate data to daily level
climate_daily = climate_df.groupby(climate_df['Date']).agg({
    "Temperature (°C)": "mean",
    "Dewpoint Temperature (°C)": "mean",
    "U Wind Component (m/s)": "mean",
    "V Wind Component (m/s)": "mean",
    "Total Precipitation (mm)": "sum",
    "Snowfall (mm)": "sum",
    "Snow Cover (%)": "mean",
}).reset_index()
climate_daily["Date"] = pd.to_datetime(climate_daily["Date"])  # Ensure datetime64[ns]


In [14]:
all_training_data = agg_daily.merge(climate_daily, on="Date", how="left")

### Get next months predictors


In [15]:
ss["Date"] = pd.to_datetime(ss["ID"].str.extract(r'(\d{4}-\d{2}-\d{2})')[0])
ss["Date"] = pd.to_datetime(ss["Date"])
ss["Source"] = ss["ID"].str.extract(r'\d{4}-\d{2}-\d{2}_(.+)')[0]
ss['v_label'] = ss['Source'].map(source_voltage_dict)
ss["consumer_device"] = ss["ID"].str.extract(r'consumer_device_(\d+)_data_user_\d+').astype(int)
ss["data_user"] = ss["ID"].str.extract(r'consumer_device_\d+_data_user_(\d+)').astype(int)
forecast = ss.merge(climate_daily, on="Date", how="left")

Capturing Seasonality and Trends: 

*    Power consumption often exhibits seasonal patterns influenced by factors like weather changes (summer vs. winter), holidays, or weekends. Features like month, quarter, and is_weekend help capture these seasonal variations

*    Features such as day_of_week and week_of_year can reveal weekly or annual trends in power usage. For instance, weekdays may have higher consumption due to industrial activity, while weekends might show reduced usage



For SARIMAX models, which are designed for time-series forecasting with seasonality and exogenous variables:

*    Handling Seasonality: SARIMAX explicitly accounts for seasonal patterns in time-series data. Features like month, quarter, and week_of_year align well with its capability to model periodic variations in power usage

*    Incorporating External Factors: SARIMAX allows the inclusion of exogenous variables (e.g., weather data). Extracted date features can act as additional external regressors to improve model performance by capturing temporal effects on consumption

*    Enhancing Moving Averages: Date features can complement the moving average component by providing context for fluctuations in power usage (e.g., spikes during holidays or weekends)

In [16]:
def extract_date_features(df, date_column='Date'):

    # Make a copy of the DataFrame to avoid modifying the original
    df = df.copy()
    
    # Convert date column to datetime format if not already
    df[date_column] = pd.to_datetime(df[date_column])
    
    # Extract date features
    df["year"] = df[date_column].dt.year
    df["month"] = df[date_column].dt.month
    df["day"] = df[date_column].dt.day
    df["day_of_week"] = df[date_column].dt.dayofweek  # Monday=0, Sunday=6
    df["week_of_year"] = df[date_column].dt.isocalendar().week
    df["quarter"] = df[date_column].dt.quarter
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)  # 1 if Sat/Sun, else 0
    
    return df

In [17]:
all_training_data = extract_date_features(all_training_data)
forecast = extract_date_features(forecast)
all_training_data['ID'] = pd.to_datetime(all_training_data['Date']).dt.strftime('%Y-%m-%d') + '_' + all_training_data['Source']

In [18]:
# Standardize the column order
columns = ['ID', 'kwh', 'Date', 'Source', 'v_label', 'consumer_device',
       'data_user', 'Temperature (°C)', 'Dewpoint Temperature (°C)',
       'U Wind Component (m/s)', 'V Wind Component (m/s)',
       'Total Precipitation (mm)', 'Snowfall (mm)', 'Snow Cover (%)', 'year',
       'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend']

In [19]:
all_training_data[columns].to_csv("Cleaned Data/all_daily_training_data.csv", index = False)
forecast[columns].to_csv("Cleaned Data/forecast_predictors.csv", index = False)

### If Merging hourly otherwise ignore

Sample submission date range:
*    Minimum date_time: 2024-09-24 00:00:00
*    Maximum date_time: 2024-10-24 00:00:00
*    Number of days: 30 + 1 (end of 10/24)
*    Number of hours: 744

The Climate dataset has
*    Minimum date_time: 2023-06-03 13:00:00
*    Maximum date_time: 2024-10-25 00:00:00
*    Number of unique date_time: 12228

The Power dataset aggregated to be hourly has
*    Minimum date_time: 2023-06-03 12:00:00
*    Maximum date_time: 2024-09-23 23:00:00
*    Number of unique date_time: 11484
*    If we include the test dates: 11484 + 31(days) * 24(hours) = 12228


**Conclusion the Climate data is 1 hour ahead.**
Shift the Climate data back one hour so that it lines up with the Power dataset and Sample submission




min_date = agg_hourly['date_time'].min()
max_date = agg_hourly['date_time'].max()
un_date = agg_hourly['date_time'].nunique()

print(f"Minimum date_time: {min_date}")
print(f"Maximum date_time: {max_date}")
print(f"Number of unique date_time: {un_date}")

Now that the data is Synchronized hourly aggregate everything to daily